AEA Extension of War and Literacy in Liberia 

In [2]:
# Importing Packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipumspy as ip
import geopandas  as gpd
from matplotlib.animation import FuncAnimation
from matplotlib.lines import Line2D
from IPython.display import HTML
import os, requests
import geopandas as gpd
from matplotlib.colors import to_rgba
from IPython.display import Video, Image
from matplotlib import animation
# unzipping the data file
from ipumspy import readers 

In [6]:
# Set local file location
wr_ddi_path = "/Users/School/GitHub/HHEI/AEA P&P/Data/idhs_wr.xml"
mr_ddi_path = '/Users/School/GitHub/HHEI/AEA P&P/Data/idhs_mr.xml'
wr_data_path = '/Users/School/GitHub/HHEI/AEA P&P/Data/WR_LBR_SL.dat.gz'
mr_data_path = '/Users/School/GitHub/HHEI/AEA P&P/Data/MR_LBR_SL.dat.gz'

Parse DDI from IPUMS-DHS website to read file. The DDI has all information on the variables chosen from data selection cart.

In [7]:
ddi_wr_obj = readers.read_ipums_ddi(wr_ddi_path)
ddi_mr_obj = readers.read_ipums_ddi(mr_ddi_path)

# Read micro data and create data frame of survey
wr_df = readers.read_microdata(ddi_wr_obj, wr_data_path)
mr_df = readers.read_microdata(ddi_mr_obj, mr_data_path)

# Check data was read correctly 
print("LBR and SL data shape", wr_df.shape)
print("LBR and SL data shape", mr_df.shape)
wr_df.head(5)


/opt/homebrew/lib/python3.14/site-packages/ipumspy/readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


LBR and SL data shape (14466, 66)
LBR and SL data shape (9289, 51)


,SAMPLE,SAMPLESTR,COUNTRY,YEAR,IDHSPID,IDHSHID,DHSID,IDHSPSU,IDHSSTRATA,CASEID,...,EDYRTOTAL,NEWSBRIG,BHCMONEY,BHCDISTANCE,NEWSFREQ_GH,DAUSHOME_GH,SONSAWAY_GH,DAUSAWAY_GH,SONSDIED_GH,DAUSDIED_GH
0,43002,43002,430,2007,43002 1 25 1,43002 1 25,LB200700000001,43002000001,4300200056,1 25 1,...,5,1,20,20,1,2,3,0,0,1
1,43002,43002,430,2007,43002 1 56 2,43002 1 56,LB200700000001,43002000001,4300200056,1 56 2,...,0,1,98,20,1,3,0,0,0,0
2,43002,43002,430,2007,43002 1122 2,43002 1122,LB200700000001,43002000001,4300200056,1122 2,...,9,11,98,20,11,1,0,0,0,0
3,43002,43002,430,2007,43002 1122 3,43002 1122,LB200700000001,43002000001,4300200056,1122 3,...,11,12,98,20,12,0,0,0,0,0
4,43002,43002,430,2007,43002 1167 2,43002 1167,LB200700000001,43002000001,4300200056,1167 2,...,98,98,98,20,98,5,0,0,0,0


In [12]:
# Build a tidy lookup of variable names and their labels from the DDI metadata
variables = pd.DataFrame(
    [
        {"variable": v.name, "label": v.label}
        for v in ddi_wr_obj.data_description
        if v.name in wr_df.columns
    ]
).sort_values("variable")

# Restrict the working sample to Liberia and Sierra Leone
country_df = wr_df.loc[wr_df["COUNTRY"].isin([430, 232])]

# For each variable in the DDI, keep only the code/label pairs that actually appear
# in the selected countries. This helps us interpret coded survey responses.
code_rows = []
for v in ddi_wr_obj.data_description:
    if v.name not in country_df.columns:
        continue

    observed_codes = set(country_df[v.name].dropna().astype(str))
    code_rows.extend(
        {"variable": v.name, "code": code, "label": label}
        for label, code in v.codes.items()
        if str(code) in observed_codes
    )

# Convert the collected code rows into a single dataframe
all_codes = pd.DataFrame(code_rows)

# Clean and Merge Data

In [16]:
# The male and female files are independent person samples, so append rows rather than joining records.
variable_pairs = {
    "AGE": ("AGE", "AGEMN"),
    "EDUCATION_LEVEL": ("EDUCLVL", "EDUCLVLMN"),
    "EDUCATION_YEARS": ("EDYRTOTAL", "EDYRTOTALMN"),
    "LITERACY": ("LIT2", "LIT2MN"),
    "LITERACY_BRIDGE": ("LITBRIG", "LITBRIGMN"),
    "LITERACY_PROGRAM_EVER": ("LITPROGRAMEV", "LITPROGRAMEVMN"),
    "WEALTH_QUINTILE": ("WEALTHQ", "WEALTHQMN"),
    "URBAN_RURAL": ("URBAN", "URBANMN"),
    "WAR_STAYED_HOME": ("MIGRWARSTAY", "MIGRWARSTAYMN"),
    "WAR_RETURNED": ("MIGRWARREL", "MIGRWARRELMN"),
    "WAR_DISPLACED_CAMP": ("MIGRWARCAMP", "MIGRWARCAMPMN"),
    "WAR_DISPLACED_BUSH": ("MIGRWARBUSH", "MIGRWARBUSHMN"),
    "WAR_DISPLACED_ABROAD": ("MIGRWARABROAD", "MIGRWARABROADMN"),
    "WAR_DISPLACED_OTHER": ("MIGRWAROTH", "MIGRWAROTHMN"),
}


def prepare_recode(data, recode, sex, source_index):
    prepared = data.copy()
    prepared["RECODE"] = recode
    prepared["SEX"] = sex
    for harmonized_name, source_names in variable_pairs.items():
        source_name = source_names[source_index]
        if source_name in prepared.columns:
            prepared[harmonized_name] = prepared[source_name]
    return prepared


combined_df = pd.concat(
    [
        prepare_recode(wr_df, "WR", "Female", 0),
        prepare_recode(mr_df, "MR", "Male", 1),
    ],
    ignore_index=True,
    sort=False,
)

country_names = {430: "Liberia", 232: "Sierra Leone"}
combined_df = combined_df.loc[combined_df["COUNTRY"].isin(country_names)].copy()
combined_df["COUNTRY_NAME"] = combined_df["COUNTRY"].map(country_names)

# The available integrated fields group counties in Liberia and districts in Sierra Leone.
liberia_mask = combined_df["COUNTRY_NAME"].eq("Liberia")
sierra_leone_mask = combined_df["COUNTRY_NAME"].eq("Sierra Leone")
combined_df["ADMIN_AREA"] = pd.Series(pd.NA, index=combined_df.index, dtype="string")
combined_df.loc[liberia_mask, "ADMIN_AREA"] = combined_df.loc[
    liberia_mask, "GEO_LR2007_2019"
].astype("string")
combined_df.loc[sierra_leone_mask, "ADMIN_AREA"] = combined_df.loc[
    sierra_leone_mask, "GEO_SL2008_2019"
].astype("string")
combined_df["ADMIN_AREA_LEVEL"] = combined_df["COUNTRY_NAME"].map(
    {"Liberia": "region (county groups)", "Sierra Leone": "region (district groups)"}
)
admin_area_labels = {
    "Liberia": {
        "1": "North Western",
        "2": "South Central, Monrovia",
        "3": "South Eastern A",
        "4": "South Eastern B",
        "5": "North Central",
    },
    "Sierra Leone": {
        "1": "Eastern",
        "2": "Northern, North Western",
        "4": "Southern",
        "5": "Western",
    },
}
combined_df["ADMIN_AREA_NAME"] = pd.Series(pd.NA, index=combined_df.index, dtype="string")
for country, labels in admin_area_labels.items():
    country_mask = combined_df["COUNTRY_NAME"].eq(country)
    area_codes = pd.to_numeric(combined_df.loc[country_mask, "ADMIN_AREA"], errors="coerce")
    combined_df.loc[country_mask, "ADMIN_AREA_NAME"] = area_codes.map(
        {int(code): label for code, label in labels.items()}
    ).astype("string")

# Full country-specific records and analysis-ready subsets with harmonized fields.
country_dfs = {
    country: frame.copy()
    for country, frame in combined_df.groupby("COUNTRY_NAME", sort=False)
}
analysis_columns = [
    "COUNTRY_NAME", "YEAR", "RECODE", "SEX", "DHSID", "ADMIN_AREA",
    "ADMIN_AREA_NAME", "ADMIN_AREA_LEVEL", *variable_pairs.keys(),
]
country_analysis_dfs = {
    country: frame.loc[:, [column for column in analysis_columns if column in frame.columns]].copy()
    for country, frame in country_dfs.items()
}

liberia_df = country_dfs["Liberia"]
sierra_leone_df = country_dfs["Sierra Leone"]
liberia_by_region = {
    area: frame.copy()
    for area, frame in liberia_df.dropna(subset=["ADMIN_AREA_NAME"]).groupby("ADMIN_AREA_NAME", sort=True)
}
sierra_leone_by_region = {
    area: frame.copy()
    for area, frame in sierra_leone_df.dropna(subset=["ADMIN_AREA_NAME"]).groupby("ADMIN_AREA_NAME", sort=True)
}

print("Combined male/female records:", combined_df.shape)
if len(liberia_df) + len(sierra_leone_df) == len(combined_df):
    print("No rows dropped")
else:
    print("rows dropped")

Combined male/female records: (23755, 123)
No rows dropped


## Filter Data - Liberia

In [25]:
import re

# Identify IPUMS codes by their DDI labels, rather than assuming numeric codes.
def ddi_codes_by_label(ddi, variable_name):
    variable = next(
        (item for item in ddi.data_description if item.name == variable_name),
        None,
    )
    if variable is None:
        return {}

    return {
        str(code): str(label).strip().lower()
        for label, code in variable.codes.items()
    }


def normalized_codes(codes, label_pattern):
    """Return codes whose DDI labels match a case-insensitive regex."""
    return {
        code
        for code, label in codes.items()
        if re.search(label_pattern, label)
    }


literacy_full_sentence_codes = {
    "Female": normalized_codes(
        ddi_codes_by_label(ddi_wr_obj, "LIT2"), r"whole sentence"
    ),
    "Male": normalized_codes(
        ddi_codes_by_label(ddi_mr_obj, "LIT2MN"), r"whole sentence"
    ),
}
if not all(literacy_full_sentence_codes.values()):
    raise ValueError("Could not identify the whole-sentence literacy code in the DDI.")

literacy_missing_codes = set().union(
    *[
        normalized_codes(
            ddi_codes_by_label(ddi, variable_name),
            r"missing|not in universe|\bniu\b|don't know|refused",
        )
        for ddi, variable_name in [
            (ddi_wr_obj, "LIT2"),
            (ddi_mr_obj, "LIT2MN"),
        ]
    ]
)

# Keep the lifelong-residence restriction, using DDI labels rather than hard-coded values.
always_codes = {
    "Female": normalized_codes(ddi_codes_by_label(ddi_wr_obj, "RESIDEINTYR"), r"\balways\b"),
    "Male": normalized_codes(ddi_codes_by_label(ddi_mr_obj, "RESIDEINTYRMN"), r"\balways\b"),
}
if not all(always_codes.values()):
    raise ValueError("Could not identify the 'Always' residence code in the DDI.")

sample = combined_df.copy()

# Keep the requested post-war survey waves.
allowed_waves = {"Liberia": {2007, 2013}, "Sierra Leone": {2008, 2013}}
sample = sample.loc[
    sample.apply(
        lambda row: row["YEAR"] in allowed_waves.get(row["COUNTRY_NAME"], set()),
        axis=1,
    )
].copy()

# Retain lifelong residents; no age restriction is applied.
sample["RESIDENCE_RESPONSE"] = pd.NA
for sex, column in [("Female", "RESIDEINTYR"), ("Male", "RESIDEINTYRMN")]:
    mask = sample["SEX"].eq(sex) & sample[column].notna()
    sample.loc[mask, "RESIDENCE_RESPONSE"] = sample.loc[mask, column].astype("string")
sample["RESIDENCE_RESPONSE"] = sample["RESIDENCE_RESPONSE"].astype("string").str.strip()
always_codes = {sex: {str(code).strip() for code in codes} for sex, codes in always_codes.items()}
permanent_resident = sample.apply(
    lambda row: str(row["RESIDENCE_RESPONSE"]) in always_codes.get(row["SEX"], set()),
    axis=1,
)

# Recode literacy: 1 for a whole sentence; 0 for other observed responses.
sample["LITERACY_CODE"] = sample["LITERACY"].astype("string").str.strip()
literacy_full_sentence_codes = {
    sex: {str(code).strip() for code in codes}
    for sex, codes in literacy_full_sentence_codes.items()
}
sample["LITERACY_BINARY"] = 0
for sex, codes in literacy_full_sentence_codes.items():
    full_sentence = sample["SEX"].eq(sex) & sample["LITERACY_CODE"].isin(codes)
    sample.loc[full_sentence, "LITERACY_BINARY"] = 1

# Drop literacy missing/NIU values, but retain all ages among lifelong residents.
literacy_missing_codes = {str(code).strip() for code in literacy_missing_codes}
has_literacy = (
    sample["LITERACY"].notna()
    & ~sample["LITERACY_CODE"].isin(literacy_missing_codes)
)
postwar_clean_df = sample.loc[permanent_resident & has_literacy].copy()

liberia_clean_df = postwar_clean_df.loc[
    postwar_clean_df["COUNTRY_NAME"].eq("Liberia")
].copy()
sierra_leone_clean_df = postwar_clean_df.loc[
    postwar_clean_df["COUNTRY_NAME"].eq("Sierra Leone")
].copy()

# Compare with the same countries and survey waves before cleaning filters.
liberia_unclean_df = sample.loc[sample["COUNTRY_NAME"].eq("Liberia")]
sierra_leone_unclean_df = sample.loc[
    sample["COUNTRY_NAME"].eq("Sierra Leone")
]
for country, unclean, clean in [
    ("Liberia", liberia_unclean_df, liberia_clean_df),
    ("Sierra Leone", sierra_leone_unclean_df, sierra_leone_clean_df),
]:
    print(
        f"{country}: wave-filtered={len(unclean):,}, retained={len(clean):,}, "
        f"removed={len(unclean) - len(clean):,}"
    )

print("Binary literacy counts among retained records:")
print(postwar_clean_df["LITERACY_BINARY"].value_counts().sort_index())

Liberia: wave-filtered=13,101, retained=6,834, removed=6,267
Sierra Leone: wave-filtered=10,654, retained=5,145, removed=5,509
Binary literacy counts among retained records:
LITERACY_BINARY
0    8216
1    3763
Name: count, dtype: int64


In [27]:
resident_eligible = permanent_resident.fillna(False)
literacy_eligible = has_literacy.fillna(False)

attrition_rows = []
for country, country_sample in sample.groupby("COUNTRY_NAME", sort=False):
    country_mask = sample["COUNTRY_NAME"].eq(country)
    resident_ok = resident_eligible.loc[country_mask]
    literacy_ok = literacy_eligible.loc[country_mask]
    attrition_rows.append(
        {
            "Country": country,
            "Wave-filtered records": len(country_sample),
            "Not lifelong resident only": int((~resident_ok & literacy_ok).sum()),
            "Missing literacy only": int((resident_ok & ~literacy_ok).sum()),
            "Fail both filters": int((~resident_ok & ~literacy_ok).sum()),
            "Retained": int((resident_ok & literacy_ok).sum()),
        }
    )

attrition_table = pd.DataFrame(attrition_rows).set_index("Country")
attrition_table

,Wave-filtered records,Not lifelong resident only,Missing literacy only,Fail both filters,Retained
Country,,,,,
Liberia,13101,6186,26,55,6834
Sierra Leone,10654,5447,31,31,5145


In [28]:
pd.crosstab(postwar_clean_df["EDUCATION_LEVEL"], postwar_clean_df["LITERACY"])
pd.crosstab([postwar_clean_df["COUNTRY_NAME"], postwar_clean_df["YEAR"], postwar_clean_df["SEX"]],
            postwar_clean_df["LITERACY_BINARY"], normalize="index")

LITERACY_BINARY                  0         1
COUNTRY_NAME YEAR SEX                       
Liberia      2007 Female  0.743226  0.256774
                  Male    0.454883  0.545117
Sierra Leone 2008 Female  0.820235  0.179765
                  Male    0.694079  0.305921

# Uppsala Conflict Data Program 

In [ ]:
# Read UCDP data from local path
ucpd_path = "/Users/School/GitHub/HHEI/Data/Public_UCDP_GED_Global.csv"
data_dir = "/Users/School/GitHub/HHEI/Data"

In [ ]:
ucpd = pd.read_csv(ucpd_path)

/var/folders/cj/dn506f5s38j4mddpf9t23jt40000gq/T/ipykernel_73559/1904420424.py:1: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  ucpd = pd.read_csv(ucpd_path)


Creating plots/ graphs 

In [ ]:
violence_labels = {1: "State-based", 2: "Non-state", 3: "One-sided"}

ged = ucpd[ucpd["country"].isin(["Liberia", "Sierra Leone"])].copy()
ged["violence_type"] = ged["type_of_violence"].map(violence_labels)

# Liberia: adm_1 = county. Sierra Leone: adm_1 = province, adm_2 = district.
ged["admin_unit"] = np.where(ged["country"] == "Liberia", ged["adm_1"], ged["adm_2"])
ged["admin_unit"] = ged["admin_unit"].fillna("Unknown / not geocoded")


,id,relid,year,active_year,code_status,type_of_violence,conflict_dset_id,conflict_new_id,conflict_name,dyad_dset_id,...,date_end,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low,gwnoa,gwnob
3277,32429,SIE-1992-3-1358-919,1992,True,Clear,3,507.0,564,NPFL - Civilians,507.0,...,1992-06-30 00:00:00.000,0,0,1,0,1,1,1,NaN,NaN
3278,32435,SIE-1991-3-1384-66,1991,True,Clear,3,532.0,571,RUF - Civilians,532.0,...,1991-09-03 00:00:00.000,0,0,1,0,1,1,1,NaN,NaN
3279,32445,SIE-1991-3-1384-60,1991,True,Clear,3,532.0,571,RUF - Civilians,532.0,...,1991-05-18 00:00:00.000,0,0,1,0,1,1,1,NaN,NaN
3280,32446,SIE-1992-3-1384-27,1992,True,Clear,3,532.0,571,RUF - Civilians,532.0,...,1992-05-31 00:00:00.000,0,0,3,0,3,3,3,NaN,NaN
3281,32447,SIE-1991-3-1384-20,1991,True,Clear,3,532.0,571,RUF - Civilians,532.0,...,1991-09-01 00:00:00.000,0,0,60,0,60,60,60,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285141,359528,GUI-2020-2-16493-0,2020,True,Clear,2,16493.0,15103,"Kpelle - Konianke, Malinké",16493.0,...,2020-03-24 00:00:00.000,0,0,0,30,30,30,30,NaN,NaN
288533,375342,GUI-2020-2-5361-0,2020,False,Clear,2,5361.0,4751,Torma - Torma Manian,5361.0,...,2020-12-27 00:00:00.000,0,0,0,17,17,17,17,NaN,NaN
299874,393367,GUI-2021-2-5576-0,2021,False,Clear,2,5576.0,4966,Kpelle - Malinké,5576.0,...,2021-04-24 00:00:00.000,0,0,0,1,1,1,1,NaN,NaN
317265,436430,SIE-2021-3-927-0,2021,False,Clear,3,80.0,460,Government of Sierra Leone - Civilians,80.0,...,2021-04-16 00:00:00.000,0,0,1,0,1,1,0,451.0,NaN


In [ ]:
country_summary = (
    ged.groupby(["country", "violence_type"])
    .agg(
        events=("id", "count"),
        deaths_best=("best", "sum"),
        deaths_low=("low", "sum"),
        deaths_high=("high", "sum"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .reset_index()
)
country_summary

In [5]:
# Larger, higher-resolution version of the conflict-event animation.
# Assumes the same imports and `ucpd` DataFrame as the original notebook
# (pandas as pd, numpy as np, geopandas as gpd, matplotlib.pyplot as plt,
#  matplotlib.animation as animation, to_rgba, Line2D, requests, os, Video, Image).

# ---- settings -------------------------------------------------------------
data_dir = "/Users/School/GitHub/HHEI/Data"
start_date, end_date = "1989-12-01", "2003-08-31"   # outer limits; frames run first event -> last event
west, east, south, north = -14.0, -7.0, 4.0, 10.6
region = ["Liberia", "Sierra Leone", "Guinea", "Ivory Coast"]
n_factions = 9        # most active factions get their own colour
step = 1              # events added per frame (1 = battle by battle; raise to speed up)
fps = 12              # frames per second

# Size / resolution (new). 10 x 11.2 in at 150 dpi = 1500 x 1680 px video:
# sharp when zoomed, and both dimensions are even, which the mp4 encoder requires.
fig_w, fig_h = 10, 11.2
video_dpi = 150
gif_dpi = 90
display_width = 900   # on-screen width in the notebook; fits a laptop screen

# ---- events, in chronological order ---------------------------------------
ev = ucpd.copy()
ev["date"] = pd.to_datetime(ev["date_start"], errors="coerce")
ev = ev[
    ev["date"].between(start_date, end_date)
    & ev["country"].isin(region)
    & ev["longitude"].between(west, east)
    & ev["latitude"].between(south, north)
].sort_values(["date", "id"]).reset_index(drop=True)

def faction(row):
    """Armed group behind the event: the first side that is not a government or civilians."""
    for side in (row["side_a"], row["side_b"]):
        s = str(side)
        if not s.startswith("Government of") and s != "Civilians":
            return s
    return str(row["side_a"])

ev["faction"] = ev.apply(faction, axis=1)
top = ev["faction"].value_counts().head(n_factions).index.tolist()
palette = ["#c0392b", "#1f618d", "#e08e0b", "#1e8449", "#7d3c98",
           "#17a2b8", "#d35487", "#7b5a3a", "#b7950b", "#34495e"]
fcolors = {f: palette[i % len(palette)] for i, f in enumerate(top)}

n = len(ev)
xy = ev[["longitude", "latitude"]].to_numpy()
rgba = np.array([to_rgba(fcolors.get(f, "#555555"), 0.65) for f in ev["faction"]])
# Slightly smaller base size so overlapping events stay distinguishable on the larger map
sizes = (10 + 3.5 * np.sqrt(ev["best"].clip(lower=0))).to_numpy()

months = pd.period_range(ev["date"].min(), ev["date"].max(), freq="M")
ev["mi"] = (ev["date"].dt.year - months[0].year) * 12 + ev["date"].dt.month - months[0].month
monthly = ev.groupby("mi").size().reindex(range(len(months)), fill_value=0)

# ---- borders --------------------------------------------------------------
def boundaries(iso, level):
    """geoBoundaries admin units, downloaded once and cached in data_dir."""
    path = os.path.join(data_dir, f"geoBoundaries-{iso}-{level}.geojson")
    if not os.path.exists(path):
        try:
            meta = requests.get(
                f"https://www.geoboundaries.org/api/current/gbOpen/{iso}/{level}/", timeout=60
            ).json()
            url = meta["gjDownloadURL"]
        except Exception:
            url = ("https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/"
                   f"gbOpen/{iso}/{level}/geoBoundaries-{iso}-{level}.geojson")
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    return gpd.read_file(path)

world = gpd.read_file(os.path.join(data_dir, "ne_50m_admin_0_countries.zip"))
focus = world[world["ADMIN"].isin(["Liberia", "Sierra Leone"])]
admin = pd.concat([boundaries("LBR", "ADM1"), boundaries("SLE", "ADM2")])

# ---- figure ---------------------------------------------------------------
fig, (ax, tl) = plt.subplots(
    2, 1, figsize=(fig_w, fig_h), gridspec_kw={"height_ratios": [8, 1], "hspace": 0.06}
)
fig.subplots_adjust(left=0.03, right=0.97, top=0.93, bottom=0.05)   # use the extra space for the map

ax.set_facecolor("#dbe9f1")
world.plot(ax=ax, color="#f3f0e8", edgecolor="#555555", linewidth=1.0)
focus.plot(ax=ax, color="#fbfaf6", edgecolor="none")
admin.boundary.plot(ax=ax, color="#a9a9a9", linewidth=0.5, zorder=2)
focus.boundary.plot(ax=ax, color="black", linewidth=1.6, zorder=2.5)
ax.set_xlim(west, east)
ax.set_ylim(south, north)
ax.set_xticks([]); ax.set_yticks([])

for name, x, y in [("GUINEA", -10.3, 10.15), ("CÔTE\nD'IVOIRE", -7.55, 7.2)]:
    ax.text(x, y, name, ha="center", va="center", fontsize=11,
            color="#8a8a8a", fontweight="bold", zorder=1)
for name, x, y in [("Freetown", -13.23, 8.48), ("Monrovia", -10.80, 6.30)]:
    ax.plot(x, y, marker="s", ms=5, color="black", zorder=6)
    ax.annotate(name, (x, y), xytext=(5, -11), textcoords="offset points", fontsize=9, zorder=6)

# Thin white outline on each dot separates events that sit on top of each other
dots = ax.scatter([], [], s=[], edgecolors="white", linewidths=0.3, zorder=3)  # all events so far
ring = ax.scatter([], [], s=[], facecolors="none", edgecolors="black",         # the newest event(s)
                  linewidths=1.5, zorder=5)
title = ax.set_title("", fontsize=12, loc="left")

handles = [Line2D([], [], marker="o", ls="", color=c, ms=8, label=f) for f, c in fcolors.items()]
handles += [
    Line2D([], [], marker="o", ls="", color="#555555", ms=8, label="Other actors"),
    Line2D([], [], marker="o", ls="", mfc="none", mec="black", mew=1.5, ms=10, label="Current event"),
]
ax.legend(handles=handles, loc="lower left", frameon=True, framealpha=0.92,
          fontsize=8.5, title="Faction", title_fontsize=9.5)

tl.bar(range(len(months)), monthly.values, color="#bbbbbb", width=1.0)
marker = tl.axvline(0, color="black", linewidth=1.5)
jan = [i for i, m in enumerate(months) if m.month == 1]
tl.set_xticks(jan)
tl.set_xticklabels([months[i].year for i in jan], fontsize=9, rotation=45)
tl.set_xlim(-0.5, len(months) - 0.5)
tl.set_ylabel("Events", fontsize=9)
tl.tick_params(axis="y", labelsize=8)
for s in ["top", "right"]:
    tl.spines[s].set_visible(False)

n_steps = int(np.ceil(n / step))
hold = fps * 3                                   # pause on the final map for 3 seconds

def update(k):
    i = min((k + 1) * step, n)                   # number of events shown so far
    j = max(i - step, 0)                         # start of the newest batch
    dots.set_offsets(xy[:i])
    dots.set_sizes(sizes[:i])
    dots.set_facecolors(rgba[:i])
    ring.set_offsets(xy[j:i])
    ring.set_sizes(sizes[j:i] * 3 + 50)
    row = ev.iloc[i - 1]
    marker.set_xdata([row["mi"], row["mi"]])
    title.set_text(
        f"{row['date'].strftime('%d %B %Y')}   ·   event {i} of {n}\n"
        f"{row['side_a']} vs {row['side_b']}   ·   "
        f"{row.get('where_coordinates', '')}, {row['country']}   ·   {int(row['best'])} deaths"
    )
    return dots, ring, marker, title

anim = animation.FuncAnimation(fig, update, frames=n_steps + hold, interval=1000 / fps)

# ---- save and show --------------------------------------------------------
out = os.path.join(data_dir, "conflict_events_large")
if animation.writers.is_available("ffmpeg"):
    anim.save(out + ".mp4", writer="ffmpeg", fps=fps, dpi=video_dpi,
              extra_args=["-pix_fmt", "yuv420p"])   # plays in browsers and QuickTime
    shown = Video(out + ".mp4", embed=True, html_attributes="controls loop", width=display_width)
else:
    anim.save(out + ".gif", writer="pillow", fps=fps, dpi=gif_dpi)
    shown = Image(filename=out + ".gif", width=display_width)
plt.close(fig)
print(f"{n} events, {ev['date'].min():%d %b %Y} to {ev['date'].max():%d %b %Y}")
shown

KeyboardInterrupt: 